# WM-811K Wafer Map — Vision Transformer

Load the dataset, filter out unknown failure types, split via `trianTestLabel`, and train a small Vision Transformer (ViT) from scratch.

In [39]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load the data

In [40]:
import pickle
import sys

# Patch old pandas module paths for compatibility
# The pickle was saved with pandas < 0.20 which used pandas.indexes.*
_compat_map = {
    'pandas.indexes':         'pandas.core.indexes',
    'pandas.indexes.base':    'pandas.core.indexes.base',
    'pandas.indexes.numeric': 'pandas.core.indexes.base',
    'pandas.indexes.range':   'pandas.core.indexes.range_',
    'pandas.indexes.multi':   'pandas.core.indexes.multi',
    'pandas.indexes.frozen':  'pandas.core.indexes.frozen',
}

for old, new in _compat_map.items():
    try:
        sys.modules[old] = __import__(new, fromlist=[''])
    except ImportError:
        pass

class _CompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        for old, new in _compat_map.items():
            if module.startswith(old):
                module = module.replace(old, new, 1)
                break
        return super().find_class(module, name)

with open("./Data/LSWMD.pkl", "rb") as f:
    try:
        df = pickle.load(f, encoding='latin1')
    except (ModuleNotFoundError, ImportError):
        f.seek(0)
        df = _CompatUnpickler(f)
        df.encoding = 'latin1'
        df = df.load()

print(f"Shape: {df.shape}")
df.head()

/var/folders/k2/4b0bgh394psfjhjl_xtj93gh000585/T/ipykernel_64534/2994757470.py:31: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  df = pickle.load(f, encoding='latin1')


Shape: (811457, 6)


,waferMap,dieSize,lotName,waferIndex,trianTestLabel,failureType
0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,1.0,[[Training]],[[none]]
1,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,2.0,[[Training]],[[none]]
2,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,3.0,[[Training]],[[none]]
3,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,4.0,[[Training]],[[none]]
4,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,5.0,[[Training]],[[none]]


## 2. Extract labels and filter out unknown failure types

The `failureType` column is stored as a nested array. Rows with an empty array (no label) or `'unknown'` are dropped — keeping only the 9 named defect classes plus `none`.

In [41]:
def extract_label(x):
    if isinstance(x, (list, np.ndarray)):
        flat = np.array(x).flatten()
        if len(flat) > 0:
            return str(flat[0])
    if isinstance(x, str):
        return x
    return "unknown"

df['failure_label'] = df['failureType'].apply(extract_label)

# Drop unlabeled rows
df_labeled = df[~df['failure_label'].isin(['unknown', ''])].reset_index(drop=True)

print(f"Before filter: {len(df):,}")
print(f"After filter:  {len(df_labeled):,}")
print(f"\nClass counts:")
print(df_labeled['failure_label'].value_counts())

Before filter: 811,457
After filter:  172,950

Class counts:
failure_label
none         147431
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64


## 3. Train / val / test split

Test set comes from `trianTestLabel == 'Test'` (the dataset's pre-assigned holdout). The `Training` rows are split 90 / 10 (stratified by class) into train and validation.

In [42]:
classes = sorted(df_labeled['failure_label'].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
NUM_CLASSES = len(classes)
print(f"{NUM_CLASSES} classes: {classes}")

df_labeled['label_idx'] = df_labeled['failure_label'].map(class_to_idx)

# 60/20/20 stratified: first carve off 20% test, then split the remaining 80% into 60/20 (i.e. 0.25 of 0.8).
y = df_labeled['label_idx'].values
idx_all = np.arange(len(df_labeled))

idx_trainval, idx_test = train_test_split(
    idx_all, test_size=0.20, stratify=y, random_state=SEED
)
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=0.25, stratify=y[idx_trainval], random_state=SEED
)

print(f"\nTrain: {len(idx_train):,}  ({len(idx_train)/len(idx_all):.1%})")
print(f"Val:   {len(idx_val):,}  ({len(idx_val)/len(idx_all):.1%})")
print(f"Test:  {len(idx_test):,}  ({len(idx_test)/len(idx_all):.1%})")

# Verify class proportions match across splits
prop_table = pd.DataFrame({
    'all':   df_labeled['failure_label'].value_counts(normalize=True),
    'train': df_labeled.iloc[idx_train]['failure_label'].value_counts(normalize=True),
    'val':   df_labeled.iloc[idx_val]['failure_label'].value_counts(normalize=True),
    'test':  df_labeled.iloc[idx_test]['failure_label'].value_counts(normalize=True),
}).fillna(0).round(4)
print("\nClass proportions per split:")
prop_table

9 classes: ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Near-full', 'Random', 'Scratch', 'none']

Train: 103,770  (60.0%)
Val:   34,590  (20.0%)
Test:  34,590  (20.0%)

Class proportions per split:


,all,train,val,test
failure_label,,,,
none,0.8524,0.8525,0.8524,0.8524
Edge-Ring,0.0560,0.0560,0.0560,0.0560
Edge-Loc,0.0300,0.0300,0.0300,0.0300
Center,0.0248,0.0248,0.0248,0.0248
Loc,0.0208,0.0208,0.0208,0.0208
Scratch,0.0069,0.0069,0.0069,0.0069
Random,0.0050,0.0050,0.0050,0.0050
Donut,0.0032,0.0032,0.0032,0.0032
Near-full,0.0009,0.0009,0.0009,0.0009


## 4. PyTorch `Dataset`

Wafer maps come in many sizes (heights/widths range widely) so each map is resized to a fixed `IMG_SIZE × IMG_SIZE` with nearest-neighbor (preserves the discrete 0/1/2 codes). The result is returned as a `(1, H, W)` float tensor in `[0, 1]` plus the integer class label.

Training instances also get random horizontal + vertical flips (each with p = 0.5). Wafer-map defects don't have a canonical orientation, so flipping is label-preserving — it effectively quadruples the variety the model sees per class, which is especially useful for the rare classes the weighted sampler keeps drawing. Validation and test stay deterministic.

In [43]:
IMG_SIZE = 64

class WaferMapDataset(Dataset):
    def __init__(self, df, indices, img_size=IMG_SIZE, augment: bool = False):
        self.maps    = df['waferMap'].values[indices]
        self.labels  = df['label_idx'].values[indices].astype(np.int64)
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def _resize(self, wmap):
        wmap = np.asarray(wmap)
        if wmap.ndim < 2 or wmap.shape[0] == 0 or wmap.shape[1] == 0:
            return np.zeros((self.img_size, self.img_size), dtype=np.float32)
        h, w = wmap.shape
        return zoom(wmap, (self.img_size / h, self.img_size / w), order=0)

    def __getitem__(self, i):
        wmap = self._resize(self.maps[i]).astype(np.float32) / 2.0  # 0/1/2 -> [0, 0.5, 1]
        if self.augment:
            # Independent p=0.5 horizontal and vertical flips. Wafer defects have no
            # canonical orientation so flips are label-preserving.
            if np.random.rand() < 0.5:
                wmap = np.ascontiguousarray(wmap[:, ::-1])
            if np.random.rand() < 0.5:
                wmap = np.ascontiguousarray(wmap[::-1, :])
        x = torch.from_numpy(wmap).unsqueeze(0)                     # (1, H, W)
        y = torch.tensor(self.labels[i], dtype=torch.long)
        return x, y

train_ds = WaferMapDataset(df_labeled, idx_train, augment=True)
val_ds   = WaferMapDataset(df_labeled, idx_val,   augment=False)
test_ds  = WaferMapDataset(df_labeled, idx_test,  augment=False)

print(f"train_ds: {len(train_ds):,}  (augmented)")
print(f"val_ds:   {len(val_ds):,}")
print(f"test_ds:  {len(test_ds):,}")

x0, y0 = train_ds[0]
print(f"\nsample x: shape={tuple(x0.shape)}, dtype={x0.dtype}, min={x0.min():.2f}, max={x0.max():.2f}")
print(f"sample y: {y0.item()} ({idx_to_class[y0.item()]})")

train_ds: 103,770  (augmented)
val_ds:   34,590
test_ds:  34,590

sample x: shape=(1, 64, 64), dtype=torch.float32, min=0.00, max=1.00
sample y: 8 (none)


## 5. `DataLoader`s

In [44]:
from torch.utils.data import WeightedRandomSampler

BATCH_SIZE  = 64
NUM_WORKERS = 0

# Per-sample weights: 1 / count[class]. Rare classes get higher draw probability,
# so each batch ends up with ~uniform class composition. Replacement=True is required
# (otherwise rare classes run out and the distribution skews back toward the majority).
train_labels = df_labeled['label_idx'].values[idx_train]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]

train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_labels),  # one "epoch" = same #steps as natural dataset
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
                          num_workers=NUM_WORKERS, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)

# --- class counts BEFORE / AFTER augmentation ---
# "Before" = raw training set (pre-sampler, no flips).
# "After"  = one epoch as the model actually sees it: WeightedRandomSampler drawing
#            with replacement, where each draw also gets an independent random
#            hflip+vflip from WaferMapDataset(augment=True).
#            We don't load images here — we just iterate sampler indices and look up
#            their labels. Flipping doesn't change a sample's class, so this is the
#            true post-augmentation class distribution.
g = torch.Generator().manual_seed(SEED)
sim_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_labels),
    replacement=True,
    generator=g,
)
sampled_idx = np.fromiter(iter(sim_sampler), dtype=np.int64, count=len(train_labels))
after_counts = np.bincount(train_labels[sampled_idx], minlength=NUM_CLASSES)

print(f"{'class':12s} {'before':>10s} {'after':>10s} {'before %':>10s} {'after %':>10s}")
print('-' * 56)
tot_b, tot_a = class_counts.sum(), after_counts.sum()
for i in range(NUM_CLASSES):
    print(f"{idx_to_class[i]:12s} "
          f"{class_counts[i]:>10,d} {after_counts[i]:>10,d} "
          f"{class_counts[i]/tot_b:>9.2%} {after_counts[i]/tot_a:>9.2%}")
print('-' * 56)
print(f"{'total':12s} {tot_b:>10,d} {tot_a:>10,d}")

# DataLoader sanity check on one batch
xb, yb = next(iter(train_loader))
print(f"\nbatch x: {tuple(xb.shape)}  dtype={xb.dtype}")
print(f"batch y: {tuple(yb.shape)}  dtype={yb.dtype}")

class            before      after   before %    after %
--------------------------------------------------------
Center            2,576     11,597     2.48%    11.18%
Donut               333     11,605     0.32%    11.18%
Edge-Loc          3,113     11,414     3.00%    11.00%
Edge-Ring         5,808     11,531     5.60%    11.11%
Loc               2,156     11,419     2.08%    11.00%
Near-full            89     11,487     0.09%    11.07%
Random              520     11,493     0.50%    11.08%
Scratch             716     11,511     0.69%    11.09%
none             88,459     11,713    85.25%    11.29%
--------------------------------------------------------
total           103,770    103,770

batch x: (64, 1, 64, 64)  dtype=torch.float32
batch y: (64,)  dtype=torch.int64


## 6. Model 3 — Vision Transformer

In [45]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def drop_path(x, drop_prob: float = 0.0, training: bool = False):
    if drop_prob == 0.0 or not training:
        return x
    keep = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    noise = torch.empty(shape, dtype=x.dtype, device=x.device).bernoulli_(keep).div_(keep)
    return x * noise


class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)


class PatchEmbed(nn.Module):
    def __init__(self, img_size=64, patch_size=8, in_channels=1, embed_dim=128):
        super().__init__()
        assert img_size % patch_size == 0
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Sequential(
            nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size),
            nn.BatchNorm2d(embed_dim),   # <-- norm right after patch projection
        )

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)  # (B, N, E)


class Block(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=2.0,
                 attn_drop=0.1, proj_drop=0.2, drop_path_rate=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(embed_dim, num_heads,
                                            dropout=attn_drop, batch_first=True)
        self.attn_drop = nn.Dropout(proj_drop)   # <-- extra drop on attn output

        self.norm2 = nn.LayerNorm(embed_dim)
        hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(proj_drop),
            nn.Linear(hidden, embed_dim),
            nn.Dropout(proj_drop),
        )
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0 else nn.Identity()

    def forward(self, x):
        h, _ = self.attn(*[self.norm1(x)] * 3, need_weights=False)
        x = x + self.drop_path(self.attn_drop(h))   # drop on attn output too
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class Model3(nn.Module):
    """
    Anti-overfit ViT for 64×64 single-channel wafer maps.

    Regularization stack:
      - BatchNorm2d after patch projection
      - pos_drop (embedding dropout)         0.15
      - attn dropout (inside MHA)            0.10
      - attn output dropout                  0.20
      - MLP dropout (×2)                     0.20
      - stochastic depth, linear 0→0.20
      - head: BN → Dropout(0.4) → Linear
    """
    def __init__(
        self,
        num_classes: int = 9,
        in_channels: int = 1,
        img_size: int = 64,
        patch_size: int = 8,
        embed_dim: int = 128,
        depth: int = 4,
        num_heads: int = 4,
        mlp_ratio: float = 2.0,
        drop_rate: float = 0.15,       # embedding + pos dropout
        attn_drop_rate: float = 0.10,  # inside MHA
        proj_drop_rate: float = 0.20,  # attn output + MLP
        drop_path_rate: float = 0.20,  # stochastic depth ceiling
        head_drop_rate: float = 0.40,  # classifier head dropout
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.pos_drop  = nn.Dropout(drop_rate)

        dpr = [r.item() for r in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio,
                  attn_drop=attn_drop_rate,
                  proj_drop=proj_drop_rate,
                  drop_path_rate=dpr[i])
            for i in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        # Stronger head: BN → heavy dropout → linear
        self.head = nn.Sequential(
            nn.BatchNorm1d(embed_dim),     # <-- stabilise GAP features
            nn.Dropout(head_drop_rate),    # <-- kill 40% before classifier
            nn.Linear(embed_dim, num_classes),
        )

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x).mean(dim=1)   # GAP
        return self.head(x)


# ---------------------------------------------------------------------------
device = (
    "cuda" if torch.cuda.is_available()
    else "mps"  if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device: {device}")

_probe = Model3(num_classes=9)
print(f"Model3 parameters: {sum(p.numel() for p in _probe.parameters()):,}")
dummy = torch.randn(4, 1, 64, 64)
print(f"output shape: {_probe(dummy).shape}")
del _probe

device: mps
Model3 parameters: 548,361
output shape: torch.Size([4, 9])


## 7. Train Model 3

In [46]:
import time

EPOCHS       = 50
LR           = 3e-4
WEIGHT_DECAY = 0.1     

model3 = Model3(num_classes=NUM_CLASSES).to(device)

with torch.no_grad():
    out = model3(xb.to(device))
print(f"forward output: {tuple(out.shape)}  (expected ({BATCH_SIZE}, {NUM_CLASSES}))")

optimizer = torch.optim.AdamW(model3.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


def run_epoch(model, loader, train: bool):
    model.train(train)
    total_loss = 0.0
    total_correct = 0
    total_seen = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss    += loss.item() * yb.size(0)
            total_correct += (logits.argmax(1) == yb).sum().item()
            total_seen    += yb.size(0)
    return total_loss / total_seen, total_correct / total_seen


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model3, train_loader, train=True)
    va_loss, va_acc = run_epoch(model3, val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss);   history['val_acc'].append(va_acc)
    print(f"epoch {epoch:2d} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} | "
          f"lr {scheduler.get_last_lr()[0]:.2e} | "
          f"{time.time()-t0:.1f}s")

forward output: (64, 9)  (expected (64, 9))
epoch  1 | train loss 1.1768 acc 0.6864 | val loss 0.9628 acc 0.8045 | lr 3.00e-04 | 35.2s
epoch  2 | train loss 0.9386 acc 0.7980 | val loss 1.0051 acc 0.7717 | lr 2.99e-04 | 34.9s
epoch  3 | train loss 0.8776 acc 0.8253 | val loss 0.8170 acc 0.8543 | lr 2.97e-04 | 34.9s
epoch  4 | train loss 0.8400 acc 0.8434 | val loss 0.9170 acc 0.8023 | lr 2.95e-04 | 34.9s
epoch  5 | train loss 0.8147 acc 0.8553 | val loss 0.8833 acc 0.8248 | lr 2.93e-04 | 34.9s
epoch  6 | train loss 0.7951 acc 0.8638 | val loss 0.7388 acc 0.8918 | lr 2.89e-04 | 35.0s
epoch  7 | train loss 0.7801 acc 0.8710 | val loss 0.8038 acc 0.8620 | lr 2.86e-04 | 34.9s
epoch  8 | train loss 0.7678 acc 0.8756 | val loss 0.8441 acc 0.8519 | lr 2.81e-04 | 35.1s
epoch  9 | train loss 0.7531 acc 0.8830 | val loss 0.7737 acc 0.8753 | lr 2.77e-04 | 35.0s
epoch 10 | train loss 0.7427 acc 0.8876 | val loss 0.7727 acc 0.8781 | lr 2.71e-04 | 142.2s
epoch 11 | train loss 0.7349 acc 0.8932 | val

## 8. Evaluate Model 3 on the test set

In [47]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, cohen_kappa_score,
)


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        ps.append(model(xb).argmax(1).cpu().numpy())
        ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ps)


y_true, y_pred = collect_predictions(model3, test_loader)
target_names = [idx_to_class[i] for i in range(NUM_CLASSES)]

print(f"Test samples: {len(y_true):,}")
print(f"\nAccuracy:           {accuracy_score(y_true, y_pred):.4f}")
print(f"Balanced accuracy:  {balanced_accuracy_score(y_true, y_pred):.4f}  (mean per-class recall)")
print(f"Cohen's kappa:      {cohen_kappa_score(y_true, y_pred):.4f}")
for avg in ('macro', 'weighted'):
    p = precision_score(y_true, y_pred, average=avg, zero_division=0)
    r = recall_score(   y_true, y_pred, average=avg, zero_division=0)
    f = f1_score(       y_true, y_pred, average=avg, zero_division=0)
    print(f"{avg:>8s}: precision {p:.4f}  recall {r:.4f}  f1 {f:.4f}")

print("\nPer-class report:")
print(classification_report(y_true, y_pred, target_names=target_names,
                            digits=4, zero_division=0))

# Confusion matrix as a labeled DataFrame (rows = true, columns = predicted)
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print("Confusion matrix (rows=true, cols=predicted):")
cm_df

Test samples: 34,590

Accuracy:           0.8820
Balanced accuracy:  0.8575  (mean per-class recall)
Cohen's kappa:      0.6634
   macro: precision 0.6514  recall 0.8575  f1 0.7127
weighted: precision 0.9471  recall 0.8820  f1 0.9062

Per-class report:
              precision    recall  f1-score   support

      Center     0.6145    0.9686    0.7519       859
       Donut     0.7068    0.8468    0.7705       111
    Edge-Loc     0.4875    0.8478    0.6191      1038
   Edge-Ring     0.9440    0.9659    0.9548      1936
         Loc     0.3608    0.7563    0.4885       718
   Near-full     0.8286    0.9667    0.8923        30
      Random     0.8229    0.8324    0.8276       173
     Scratch     0.1013    0.6527    0.1754       239
        none     0.9960    0.8804    0.9347     29486

    accuracy                         0.8820     34590
   macro avg     0.6514    0.8575    0.7127     34590
weighted avg     0.9471    0.8820    0.9062     34590

Confusion matrix (rows=true, cols=predicte

,Center,Donut,Edge-Loc,Edge-Ring,Loc,Near-full,Random,Scratch,none
Center,832,3,2,1,17,0,0,1,3
Donut,3,94,1,0,10,0,2,0,1
Edge-Loc,9,0,880,23,65,0,1,28,32
Edge-Ring,6,1,44,1870,0,0,1,4,10
Loc,22,18,41,0,543,0,2,64,28
Near-full,0,0,0,0,1,29,0,0,0
Random,7,3,4,0,8,6,144,0,1
Scratch,2,5,8,1,38,0,1,156,28
none,473,9,825,86,823,0,24,1287,25959
